In [38]:
from PIL import Image
import torch
from diffusers import AutoPipelineForInpainting
from diffusers.utils import load_image, make_image_grid
import numpy as np
from PIL import Image
import random
import math
import glob

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size
    
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

def generate_and_display(prompt, image, mask_image, pipeline):
    num_images = 3
    prompt = [prompt] * num_images

    images = pipeline(prompt, image=image, mask_image=mask_image).images

    grid = image_grid([image, mask_image] + images, rows=1, cols=5)
    display(grid)

def generate_random_masks(image, aspect_ratio, area, num_masks=1):
    image_width, image_height = image.size
    
    if 0 <= area <= 1:
        mask_area = int(area * image_width * image_height)
    else:
        mask_area = int(area)
    
    mask_height = int(math.sqrt(mask_area / aspect_ratio))
    mask_width = int(mask_height * aspect_ratio)

    mask_height = max(1, mask_height)
    mask_width = max(1, mask_width)
    
    if mask_width > image_width or mask_height > image_height:
        raise ValueError(f"Маска с размерами ({mask_width}x{mask_height}) слишком большая для изображения ({image_width}x{image_height}) с заданным соотношением сторон.")
    
    mask = np.zeros((image_height, image_width), dtype=np.uint8)
    for _ in range(num_masks):
        x_pos = random.randint(0, image_width - mask_width)
        y_pos = random.randint(0, image_height - mask_height)
        
        mask[y_pos:y_pos+mask_height, x_pos:x_pos+mask_width] = 1
        
    mask = Image.fromarray(mask * 255)
        
    return mask

In [39]:
images = []
masks_middle = []
masks_long = []
masks_big = []
for img_path in sorted(glob.glob("/home/anmilka/different/test_images/*.png") + glob.glob("/home/anmilka/different/test_images/*.jpg")):
    image = Image.open(img_path)
    image = image.resize((512, 512))
    images.append(image)    

    mask_middle = generate_random_masks(image, 1, 0.1, 3)
    mask_long = generate_random_masks(image, 70, 0.01, 5)
    mask_big = generate_random_masks(image, 1, 0.4, 1)

    masks_middle.append(mask_middle)
    masks_long.append(mask_long)
    masks_big.append(mask_big)

prompts = [
    "girl sings on stage, high quality, realism",
    "crowd of people, high quality, realism",
    "page of a book, macro, high quality, realism",
    "cartoone rick and morty",
    "poppy flower, macro, high quality, realism",
    "Vienna street",
    "coffee house, high quality, realism",
    "longhair dachshund, high quality, realism",
    "mountain landscape, high quality, realism",
    "two girls and a man, high quality, realism",
    "outer space, high quality, realism",
]

In [40]:
stable_diffusion = AutoPipelineForInpainting.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16, variant="fp16"
).to("cuda:3")
stable_diffusion.enable_model_cpu_offload()


for i, image in enumerate(images):
    print(prompts[i])
    generate_and_display(prompts[i], image, masks_middle[i], stable_diffusion)
    generate_and_display(prompts[i], image, masks_long[i], stable_diffusion)
    generate_and_display(prompts[i], image, masks_big[i], stable_diffusion)
    print("\n\n")


Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  6.10it/s]


girl sings on stage, high quality, realism


100%|██████████| 50/50 [00:08<00:00,  5.63it/s]


100%|██████████| 50/50 [00:08<00:00,  5.58it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


100%|██████████| 50/50 [00:09<00:00,  5.49it/s]


...

100%|██████████| 50/50 [00:09<00:00,  5.53it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.





outer space, high quality, realism


100%|██████████| 50/50 [00:08<00:00,  5.63it/s]


100%|██████████| 50/50 [00:08<00:00,  5.60it/s]


100%|██████████| 50/50 [00:09<00:00,  5.53it/s]


In [41]:
stable_diffusion = AutoPipelineForInpainting.from_pretrained(
    "kandinsky-community/kandinsky-2-1-inpaint", torch_dtype=torch.float16
).to("cuda:3")
stable_diffusion.enable_model_cpu_offload()


for i, image in enumerate(images):
    print(prompts[i])
    generate_and_display(prompts[i], image, masks_middle[i], stable_diffusion)
    generate_and_display(prompts[i], image, masks_long[i], stable_diffusion)
    generate_and_display(prompts[i], image, masks_big[i], stable_diffusion)
    print("\n\n")

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  3.29it/s]


girl sings on stage, high quality, realism


100%|██████████| 100/100 [00:18<00:00,  5.51it/s]


100%|██████████| 100/100 [00:17<00:00,  5.69it/s]


100%|██████████| 100/100 [00:17<00:00,  5.83it/s]


...

100%|██████████| 100/100 [00:17<00:00,  5.79it/s]





outer space, high quality, realism


100%|██████████| 100/100 [00:17<00:00,  5.78it/s]


100%|██████████| 100/100 [00:17<00:00,  5.66it/s]


100%|██████████| 100/100 [00:17<00:00,  5.71it/s]


In [42]:
stable_diffusion = AutoPipelineForInpainting.from_pretrained(
    "prompthero/openjourney", torch_dtype=torch.float16
).to("cuda:3")
stable_diffusion.enable_model_cpu_offload()


for i, image in enumerate(images):
    print(prompts[i])
    generate_and_display(prompts[i], image, masks_middle[i], stable_diffusion)
    generate_and_display(prompts[i], image, masks_long[i], stable_diffusion)
    generate_and_display(prompts[i], image, masks_big[i], stable_diffusion)
    print("\n\n")

Loading pipeline components...: 100%|██████████| 7/7 [00:03<00:00,  2.29it/s]


girl sings on stage, high quality, realism


100%|██████████| 50/50 [00:08<00:00,  5.56it/s]


100%|██████████| 50/50 [00:08<00:00,  5.59it/s]


100%|██████████| 50/50 [00:09<00:00,  5.47it/s]


...

100%|██████████| 50/50 [00:09<00:00,  5.55it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.





outer space, high quality, realism


100%|██████████| 50/50 [00:08<00:00,  5.56it/s]


100%|██████████| 50/50 [00:09<00:00,  5.51it/s]


100%|██████████| 50/50 [00:09<00:00,  5.51it/s]


In [4]:
import torch
from controlnet_aux import CannyDetector
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.utils import load_image, make_image_grid
from PIL import Image

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", 
    torch_dtype=torch.float16,
    varient="fp16")

def generate_mixed_image(pipeline, prompt, img, canny_img, ip_adapter_image):
    pipeline.set_ip_adapter_scale(0.5)
    
    generated_images = pipeline(prompt = prompt, 
                height = 512, 
                width = 512,
                ip_adapter_image = ip_adapter_image,
                image = canny_img,
                guidance_scale = 6,
                controlnet_conditioning_scale = 0.7,
                num_inference_steps = 50,
                num_images_per_prompt = 3).images

    generated_images = [img] + generated_images
    grid = make_image_grid(generated_images, cols=4, rows=1)
    display(grid)

In [5]:
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet, 
    torch_dtype=torch.float16)

pipe.load_ip_adapter("h94/IP-Adapter", 
                     subfolder="models", 
                     weight_name="ip-adapter_sd15.bin")
 
pipe.enable_model_cpu_offload()

Loading pipeline components...: 100%|██████████| 7/7 [00:02<00:00,  2.80it/s]


In [19]:
img = Image.open("/home/anmilka/different/image12.png")
img = img.resize((512, 512))
display(img)
print(len(image.split()))
canny = CannyDetector()
canny_img = canny(img, detect_resolution=512, image_resolution=512)

display(canny_img)

3


In [10]:
prompt = "young woman on a Vienna street, high quality, realism"
generate_mixed_image(pipe, prompt, img, canny_img, images[5])

prompt = "woman in a coffee shop, high quality, realism"
generate_mixed_image(pipe, prompt, img, canny_img, images[6])

100%|██████████| 50/50 [00:13<00:00,  3.59it/s]


100%|██████████| 50/50 [00:12<00:00,  4.03it/s]


In [12]:
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "prompthero/openjourney",
    controlnet=controlnet, 
    torch_dtype=torch.float16)

pipe.load_ip_adapter("h94/IP-Adapter", 
                     subfolder="models", 
                     weight_name="ip-adapter_sd15.bin")
 
pipe.enable_model_cpu_offload()

prompt = "young woman on a Vienna street, high quality, realism"
generate_mixed_image(pipe, prompt, img, canny_img, images[5])

prompt = "woman in a coffee shop, high quality, realism"
generate_mixed_image(pipe, prompt, img, canny_img, images[6])

100%|██████████| 50/50 [00:12<00:00,  3.91it/s]


100%|██████████| 50/50 [00:13<00:00,  3.83it/s]


In [37]:
from diffusers import KandinskyPriorPipeline, KandinskyPipeline
from diffusers.utils import load_image
import PIL

import torch

pipe_prior = KandinskyPriorPipeline.from_pretrained(
    "kandinsky-community/kandinsky-2-1-prior", torch_dtype=torch.float16
)
pipe_prior.to("cuda:3")
pipe = KandinskyPipeline.from_pretrained("kandinsky-community/kandinsky-2-1", torch_dtype=torch.float16)
pipe.to("cuda:3")

images_texts = ["young woman on a Vienna street, high quality, realism", img, images[5]]
weights = [0.2, 0.5, 0.1]
prompt = ""
prior_out = pipe_prior.interpolate(images_texts, weights)

generated_images = pipe(prompt, **prior_out, height=512, width=512, num_images_per_prompt = 3).images

generated_images = [img] + generated_images
grid = make_image_grid(generated_images, cols=4, rows=1)
display(grid)


100%|██████████| 100/100 [00:12<00:00,  7.78it/s]


In [35]:
images_texts = ["woman in a coffee shop, high quality, realism", img, images[6]]
weights = [0.2, 0.5, 0.1]
prompt = ""
prior_out = pipe_prior.interpolate(images_texts, weights)

generated_images = pipe(prompt, **prior_out, height=512, width=512, num_images_per_prompt = 3).images

generated_images = [img] + generated_images
grid = make_image_grid(generated_images, cols=4, rows=1)
display(grid)

100%|██████████| 100/100 [00:12<00:00,  7.81it/s]
